# Important Data Science Topics

This separate notebook covers important data science ideas that students should learn after basic Python, cleaning, preprocessing, and modeling.

It uses the same file:

`student_performance_test_dataset.csv`

Topics included:

1. Data science project lifecycle
2. Descriptive statistics
3. Sampling and bias
4. Correlation vs causation
5. GroupBy analysis
6. Joins and merging data
7. Probability basics
8. Confidence intervals
9. Hypothesis testing
10. Data leakage
11. Class imbalance
12. Choosing evaluation metrics
13. Feature importance
14. Regularization
15. Error analysis
16. Reproducibility
17. Ethics and privacy
18. Final project checklist

## 1. Setup

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    try:
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic("pip", "install " + " ".join(missing_packages))
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    except Exception as error:
        print("Please run this in a notebook cell instead:")
        print("%pip install numpy pandas matplotlib scikit-learn")
        raise error
else:
    print("All required packages are already installed.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", 80)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Load the Dataset

In [ ]:
df = pd.read_csv("student_performance_test_dataset.csv")
print("Rows and columns:", df.shape)
df.head()

## 3. Data Science Project Lifecycle

A complete data science project is not only model training.

Typical lifecycle:

1. Business or research question
2. Data collection
3. Data understanding
4. Data cleaning
5. Feature engineering
6. Modeling
7. Evaluation
8. Communication
9. Deployment or decision support
10. Monitoring and improvement

Classroom example question: Which factors are associated with student exam performance?

## 4. Descriptive Statistics

Descriptive statistics summarize a dataset.

Important measures:

- Mean: average
- Median: middle value
- Mode: most common value
- Standard deviation: spread of values
- Minimum and maximum: range
- Percentiles: values below which a percentage of data falls

In [ ]:
numeric_columns = ["study_hours", "attendance_rate", "previous_score", "sleep_hours", "final_score"]
df[numeric_columns].agg(["mean", "median", "std", "min", "max"]).round(2)

In [ ]:
# Percentiles help us understand low, typical, and high values.
df[numeric_columns].quantile([0.25, 0.50, 0.75, 0.90]).round(2)

## 5. Sampling and Bias

A sample should represent the population we care about.

Examples of bias:

- Only collecting data from high-performing students
- Missing students without internet access
- Surveying only students who volunteer
- Using last year's students when this year's curriculum changed

A biased sample can produce misleading conclusions.

In [ ]:
# Compare a random sample with a biased sample.
random_sample = df.sample(80, random_state=RANDOM_SEED)
biased_sample = df.sort_values("study_hours", ascending=False).head(80)

comparison = pd.DataFrame(
    {
        "full_data_mean": df[numeric_columns].mean(),
        "random_sample_mean": random_sample[numeric_columns].mean(),
        "biased_sample_mean": biased_sample[numeric_columns].mean(),
    }
).round(2)

comparison

## 6. Correlation Is Not Causation

Correlation means two variables move together.

Causation means one variable directly causes a change in another.

Example: study hours may be correlated with final score. But to prove study hours cause higher scores, we need stronger evidence, such as experiments or careful causal analysis.

In [ ]:
correlation_with_score = df[numeric_columns].corr()["final_score"].sort_values(ascending=False)
correlation_with_score.round(3)

## 7. GroupBy Analysis

`groupby` is one of the most important pandas tools. It helps answer questions by category.

In [ ]:
# Average score and pass rate by internet access.
df.groupby("internet_access").agg(
    student_count=("student_id", "count"),
    average_score=("final_score", "mean"),
    pass_rate=("passed", "mean"),
).round(2)

In [ ]:
# Average score by parent education and extra classes.
pd.pivot_table(
    df,
    values="final_score",
    index="parent_education",
    columns="extra_classes",
    aggfunc="mean",
).round(1)

## 8. Joins and Merging Data

Real projects often combine multiple tables.

Common join types:

- Inner join: keep matching rows only
- Left join: keep all rows from the left table
- Right join: keep all rows from the right table
- Outer join: keep all rows from both tables

In [ ]:
# Create a small second table to demonstrate merging.
student_support = pd.DataFrame(
    {
        "student_id": [1, 2, 3, 4, 5, 999],
        "counseling_sessions": [2, 0, 1, 3, 1, 5],
        "library_visits": [10, 3, 5, 8, 4, 7],
    }
)

merged_left = df.merge(student_support, on="student_id", how="left")
merged_inner = df.merge(student_support, on="student_id", how="inner")

print("Left join shape:", merged_left.shape)
print("Inner join shape:", merged_inner.shape)
merged_left.head()

## 9. Probability Basics

Probability measures how likely something is.

Example questions:

- What is the probability that a randomly selected student passed?
- What is the probability that a student passed given they attended extra classes?

In [ ]:
prob_pass = df["passed"].mean()
prob_pass_extra = df.loc[df["extra_classes"] == "Yes", "passed"].mean()
prob_pass_no_extra = df.loc[df["extra_classes"] == "No", "passed"].mean()

print("P(pass):", round(prob_pass, 3))
print("P(pass | extra classes = Yes):", round(prob_pass_extra, 3))
print("P(pass | extra classes = No):", round(prob_pass_no_extra, 3))

## 10. Confidence Intervals

A confidence interval gives a range of plausible values for a population statistic.

Here we calculate an approximate 95% confidence interval for the average final score.

In [ ]:
scores = df["final_score"].dropna()
mean_score = scores.mean()
standard_error = scores.std(ddof=1) / np.sqrt(len(scores))
lower = mean_score - 1.96 * standard_error
upper = mean_score + 1.96 * standard_error

print("Mean final score:", round(mean_score, 2))
print("Approximate 95% confidence interval:", round(lower, 2), "to", round(upper, 2))

## 11. Simple Hypothesis Testing Idea

A hypothesis test asks whether an observed difference may be due to random chance.

Example question: Do students with extra classes have a different average score than students without extra classes?

This notebook shows the idea using a permutation test. A permutation test repeatedly shuffles group labels and compares the observed difference with random differences.

In [ ]:
extra_yes = df.loc[df["extra_classes"] == "Yes", "final_score"].dropna()
extra_no = df.loc[df["extra_classes"] == "No", "final_score"].dropna()
observed_difference = extra_yes.mean() - extra_no.mean()

combined_scores = pd.concat([extra_yes, extra_no]).to_numpy()
group_size = len(extra_yes)
random_differences = []

for _ in range(1000):
    shuffled = np.random.permutation(combined_scores)
    random_difference = shuffled[:group_size].mean() - shuffled[group_size:].mean()
    random_differences.append(random_difference)

p_value = np.mean(np.abs(random_differences) >= abs(observed_difference))

print("Observed difference:", round(observed_difference, 2))
print("Approximate permutation p-value:", round(p_value, 3))

## 12. Data Leakage

Data leakage happens when information from the future or target accidentally enters the model.

Example: using `final_score` to predict `passed` is leakage because `passed` is created from `final_score`.

Leakage makes test results look excellent but fail in the real world.

In [ ]:
# Bad example: final_score leaks the answer because passed is based on final_score.
leaky_features = ["study_hours", "attendance_rate", "previous_score", "sleep_hours", "final_score"]
safe_features = ["study_hours", "attendance_rate", "previous_score", "sleep_hours"]

def simple_numeric_model_score(features):
    data = df[features + ["passed"]].dropna()
    X = data[features]
    y = data["passed"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

print("Safe model accuracy:", round(simple_numeric_model_score(safe_features), 3))
print("Leaky model accuracy:", round(simple_numeric_model_score(leaky_features), 3))

## 13. Class Imbalance

Class imbalance means one class is much more common than another.

If 95% of students pass, a model that always predicts pass gets 95% accuracy but is not useful for finding students who need help.

For imbalanced data, look beyond accuracy.

In [ ]:
class_counts = df["passed"].value_counts().sort_index()
class_percent = (df["passed"].value_counts(normalize=True).sort_index() * 100).round(2)

pd.DataFrame({"count": class_counts, "percent": class_percent})

## 14. Choosing Evaluation Metrics

Different questions need different metrics.

- Accuracy: overall correctness
- Precision: when false positives are costly
- Recall: when false negatives are costly
- F1 score: balance between precision and recall

For student support, recall may be important because missing a struggling student can be costly.

In [ ]:
model_data = df.dropna(subset=["study_hours", "attendance_rate", "previous_score", "sleep_hours", "passed"]).copy()
X = model_data[["study_hours", "attendance_rate", "previous_score", "sleep_hours", "internet_access", "parent_education", "extra_classes"]]
y = model_data["passed"]

numeric_features = ["study_hours", "attendance_rate", "previous_score", "sleep_hours"]
categorical_features = ["internet_access", "parent_education", "extra_classes"]

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)]), categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

classifier.fit(X_train, y_train)
pred = classifier.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall:", round(recall_score(y_test, pred), 3))
print("F1:", round(f1_score(y_test, pred), 3))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred))

## 15. Feature Importance

Feature importance helps explain which inputs were useful to a model.

Important caution: feature importance is not always causal.

In [ ]:
forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)),
    ]
)

forest.fit(X_train, y_train)

feature_names = forest.named_steps["preprocessor"].get_feature_names_out()
importances = forest.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame(
    {"feature": feature_names, "importance": importances}
).sort_values("importance", ascending=False)

feature_importance.head(10)

## 16. Regularization

Regularization prevents models from becoming too complex.

In simple terms, it discourages very large coefficients.

This often improves performance on new data.

In [ ]:
# Ridge regression is linear regression with regularization.
# Here we predict final_score from numeric features as a demonstration.
ridge_data = df[safe_features + ["final_score"]].dropna()
X_ridge = ridge_data[safe_features]
y_ridge = ridge_data["final_score"]

for alpha in [0.1, 1, 10, 100]:
    ridge_model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha)),
        ]
    )
    scores = cross_val_score(ridge_model, X_ridge, y_ridge, cv=5, scoring="r2")
    print("alpha =", alpha, "mean R2 =", round(scores.mean(), 3))

## 17. Error Analysis

After modeling, inspect wrong predictions.

Error analysis can reveal:

- Missing features
- Data quality problems
- Groups where the model performs poorly
- Cases that are genuinely difficult

In [ ]:
error_table = X_test.copy()
error_table["actual"] = y_test.values
error_table["predicted"] = pred
error_table["correct"] = error_table["actual"] == error_table["predicted"]

error_table[error_table["correct"] == False].head(10)

## 18. Reproducibility

Reproducibility means someone else can run your work and get the same result.

Good habits:

- Set random seeds
- Save data versions
- Record package versions
- Keep notebooks organized
- Avoid manual hidden steps
- Use clear file names
- Explain assumptions

In [ ]:
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)

import sklearn
print("scikit-learn version:", sklearn.__version__)

## 19. Ethics and Privacy

Data science has real-world impact.

Important questions:

- Do we have permission to use this data?
- Does the dataset include private information?
- Could the model unfairly affect a group of people?
- Are predictions being used as support or as final judgment?
- Can a student or user understand and challenge the decision?

For education data, predictions should support teachers and students, not replace human judgment.

## 20. Final Project Checklist

Use this checklist before submitting a data science project:

1. Clear problem statement
2. Data source explained
3. Data dictionary included
4. Missing values handled
5. Duplicates checked
6. Outliers investigated
7. Categorical variables encoded
8. Numeric features scaled when needed
9. Train-test split used correctly
10. Data leakage avoided
11. Correct metrics selected
12. Model compared to a simple baseline
13. Results explained clearly
14. Limitations documented
15. Ethical risks considered
16. Code can be rerun from top to bottom

## 21. Suggested Student Mini Projects

1. Build a pass/fail prediction model and explain which metric matters most.
2. Create a student support dashboard using groupby tables and plots.
3. Compare biased and random samples from the dataset.
4. Write a short report explaining correlation vs causation.
5. Find wrong predictions and suggest new features that might help.
6. Build a reproducible project folder with data, notebook, model, and report.